In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
  !pip install numpy numba deap

In [3]:
import random
import numpy as np
from numba import njit
from deap import base, creator, tools
from collections import defaultdict
from copy import deepcopy
import math

# ==========================================
# PART 0: ADVANCED DATA STRUCTURES
# ==========================================

class EliteArchive:
    """External archive of Pareto-optimal solutions (Point 9)."""
    def __init__(self, max_size=100):
        self.archive = []
        self.max_size = max_size
        
    def add(self, individual, fitness_values):
        """Add individual if it's non-dominated."""
        is_dominated = False
        to_remove = []
        
        for i, (archived_ind, archived_fit) in enumerate(self.archive):
            # Check if new individual dominates archived one
            if self._dominates(fitness_values, archived_fit):
                to_remove.append(i)
            # Check if archived dominates new individual
            elif self._dominates(archived_fit, fitness_values):
                is_dominated = True
                break
        
        if not is_dominated:
            for i in sorted(to_remove, reverse=True):
                del self.archive[i]
            self.archive.append((deepcopy(individual), fitness_values))
            
            # Maintain size limit via crowding distance
            if len(self.archive) > self.max_size:
                self._prune_by_crowding()
    
    def _dominates(self, fit1, fit2):
        """Check if fit1 dominates fit2 (maximizing all objectives)."""
        return all(f1 >= f2 for f1, f2 in zip(fit1, fit2)) and any(f1 > f2 for f1, f2 in zip(fit1, fit2))
    
    def _prune_by_crowding(self):
        """Remove least crowded individual (crowding distance metric)."""
        if len(self.archive) <= 1:
            return
        
        distances = [0] * len(self.archive)
        
        # For each objective, sort and calculate distance
        for obj_idx in range(3):  # 3 objectives: NL, DU, weak_count
            sorted_indices = sorted(range(len(self.archive)), 
                                  key=lambda i: self.archive[i][1][obj_idx])
            
            # Boundary points get infinite distance
            distances[sorted_indices[0]] = float('inf')
            distances[sorted_indices[-1]] = float('inf')
            
            # Interior points
            f_max = self.archive[sorted_indices[-1]][1][obj_idx]
            f_min = self.archive[sorted_indices[0]][1][obj_idx]
            
            if f_max - f_min > 0:
                for i in range(1, len(sorted_indices) - 1):
                    idx = sorted_indices[i]
                    prev_f = self.archive[sorted_indices[i-1]][1][obj_idx]
                    next_f = self.archive[sorted_indices[i+1]][1][obj_idx]
                    distances[idx] += (next_f - prev_f) / (f_max - f_min)
        
        # Remove least crowded
        min_dist_idx = distances.index(min(distances))
        del self.archive[min_dist_idx]
    
    def get_random(self):
        """Return random elite from archive."""
        if self.archive:
            return deepcopy(random.choice(self.archive)[0])
        return None
    
    def sample_multiple(self, n):
        """Sample n random individuals from archive."""
        if not self.archive:
            return []
        return [deepcopy(random.choice(self.archive)[0]) for _ in range(min(n, len(self.archive)))]
    
    def get_best_by_nl(self):
        """Return individual with highest NL from archive."""
        if not self.archive:
            return None
        best_ind = max(self.archive, key=lambda x: x[1][0])
        return deepcopy(best_ind[0])


class TabuMemory:
    """Tabu list to avoid re-discovering solutions (Point 11)."""
    def __init__(self, max_size=50):
        self.tabu_list = []
        self.max_size = max_size
    
    def add(self, individual_tuple):
        """Add individual hash to tabu list."""
        self.tabu_list.append(individual_tuple)
        if len(self.tabu_list) > self.max_size:
            self.tabu_list.pop(0)
    
    def is_tabu(self, individual_tuple):
        """Check if individual is in tabu list."""
        return individual_tuple in self.tabu_list
    
    def clear(self):
        """Clear tabu list."""
        self.tabu_list = []


class IslandPopulation:
    """Island model for multi-population evolution (Point 12)."""
    def __init__(self, num_islands=3, pop_per_island=200):
        self.islands = [[] for _ in range(num_islands)]
        self.num_islands = num_islands
        self.pop_per_island = pop_per_island
        self.migration_interval = 10
    
    def get_all_individuals(self):
        """Flatten all islands."""
        result = []
        for island in self.islands:
            result.extend(island)
        return result
    
    def set_islands(self, individuals):
        """Distribute individuals across islands."""
        self.islands = [[] for _ in range(self.num_islands)]
        for i, ind in enumerate(individuals):
            self.islands[i % self.num_islands].append(ind)
    
    def migrate(self, elite_ratio=0.05):
        """Exchange elite individuals between islands."""
        elite_per_island = max(1, int(self.pop_per_island * elite_ratio))
        
        # Collect elites from each island
        elites_per_island = []
        for island in self.islands:
            island.sort(key=lambda x: x.fitness.values, reverse=True)
            elites_per_island.append([deepcopy(e) for e in island[:elite_per_island]])
        
        # Rotate elites to neighboring islands
        for i in range(self.num_islands):
            next_i = (i + 1) % self.num_islands
            # Replace worst individuals with elites from previous island
            self.islands[i].sort(key=lambda x: x.fitness.values, reverse=True)
            for j in range(min(elite_per_island, len(self.islands[i]))):
                self.islands[i][-(j+1)] = elites_per_island[i][j]


# ==========================================
# PART 1: JIT-COMPILED CRYPTOGRAPHIC ENGINES
# ==========================================

@njit
def fwht_fast(a):
    """Fast Walsh-Hadamard Transform - O(N log N) compiled to machine code."""
    h = 1
    n = len(a)
    while h < n:
        for i in range(0, n, h * 2):
            for j in range(i, i + h):
                x = a[j]
                y = a[j + h]
                a[j] = x + y
                a[j + h] = x - y
        h *= 2
    return a

@njit
def compute_differential_uniformity_fast(sbox):
    """JIT-compiled DDT calculator. Returns both the max DU and the full table."""
    n = len(sbox)
    ddt = np.zeros((n, n), dtype=np.int32)
    max_du = 0
    
    for dx in range(1, n):
        for x in range(n):
            dy = sbox[x] ^ sbox[x ^ dx]
            ddt[dx, dy] += 1
            if ddt[dx, dy] > max_du:
                max_du = ddt[dx, dy]
                
    return max_du, ddt

@njit
def fast_dot_product(a, b):
    """Calculates bitwise dot product over GF(2) without using Python string methods."""
    res = a & b
    res ^= res >> 4
    res ^= res >> 2
    res ^= res >> 1
    return res & 1

@njit
def evaluate_nl_fast(sbox):
    """JIT-compiled Non-Linearity evaluator using the Walsh-Hadamard Transform."""
    num_inputs = 256
    min_nl = 128
    count_at_min = 0

    for v in range(1, 256):
        a = np.empty(num_inputs, dtype=np.int32)
        for x in range(num_inputs):
            dot = fast_dot_product(sbox[x], v)
            a[x] = 1 if dot == 0 else -1

        fwht_result = fwht_fast(a.copy())
        
        max_walsh_value = 0
        for val in fwht_result:
            if abs(val) > max_walsh_value:
                max_walsh_value = abs(val)
                
        nl = 128 - (max_walsh_value // 2)
        
        if nl < min_nl:
            min_nl = nl
            count_at_min = 1
        elif nl == min_nl:
            count_at_min += 1
            
    return min_nl, count_at_min

@njit
def compute_behavioral_distance(sbox1, sbox2):
    """Compute behavioral distance based on DDT differences (Point 8)."""
    _, ddt1 = compute_differential_uniformity_fast(sbox1)
    _, ddt2 = compute_differential_uniformity_fast(sbox2)
    
    # Frobenius norm of DDT difference
    distance = 0.0
    for i in range(len(ddt1)):
        for j in range(len(ddt1[0])):
            distance += ((ddt1[i, j] - ddt2[i, j]) ** 2)
    
    return math.sqrt(distance)

# ==========================================
# PART 2: ADAPTIVE FITNESS AND MUTATIONS
# ==========================================

class AdaptiveGeneticOperators:
    """Manages adaptive genetic operators (Points 1, 3, 4)."""
    
    def __init__(self):
        self.generation = 0
        self.initial_best_fitness = None
        self.best_fitness_history = []
        self.population_diversity_history = []
        self.current_cx_pb = 0.15
        self.current_mut_pb = 0.85
        self.current_weights = (6.0, 1.0, 0.5)
        self.phase = "early"  # IMPROVEMENT: Track phase explicitly
    
    def update_adaptive_parameters(self, generation, pop_diversity, best_fitness, population_size):
        """Update adaptive mutation rate and weights (Points 1, 3)."""
        self.generation = generation
        self.best_fitness_history.append(best_fitness)
        self.population_diversity_history.append(pop_diversity)
        
        # IMPROVEMENT: Phase-based weight strategy - prioritize NL more aggressively
        total_gens = 1000
        phase = generation / total_gens
        
        if phase < 0.33:
            # Early: AGGRESSIVE NL focus
            self.current_weights = (8.0, 0.5, 0.2)
            self.phase = "early"
        elif phase < 0.66:
            # Mid: Balance NL and DU, reduce weak-count focus
            self.current_weights = (6.0, 1.5, 0.3)
            self.phase = "mid"
        else:
            # Late: Fine-tune with modest DU consideration
            self.current_weights = (4.0, 1.0, 0.2)
            self.phase = "late"
        
        # Adaptive mutation rate with AGGRESSIVE escalation for NL plateaus
        if pop_diversity < 0.12:  # Stagnation detected
            self.current_mut_pb = min(0.92, self.current_mut_pb + 0.08)
        elif pop_diversity > 0.55:  # High diversity
            self.current_mut_pb = max(0.60, self.current_mut_pb - 0.05)
        
        # Check if NL improvement is stalling
        if len(self.best_fitness_history) > 15:
            recent_improvement = self.best_fitness_history[-1] - self.best_fitness_history[-15]
            if recent_improvement < 0.5:  # Very slow improvement - trigger exploration boost
                self.current_mut_pb = min(0.95, self.current_mut_pb + 0.05)
        
        self.current_cx_pb = 0.25 if pop_diversity < 0.2 else 0.12
    
    def get_weights(self):
        """Return current adaptive weights."""
        return self.current_weights
    
    def get_mutation_rate(self):
        """Return current mutation rate."""
        return self.current_mut_pb
    
    def get_crossover_rate(self):
        """Return current crossover rate."""
        return self.current_cx_pb
    
    def calculate_hamming_distance(self, ind1, ind2):
        """Calculate normalized Hamming distance."""
        return sum(a != b for a, b in zip(ind1, ind2)) / len(ind1)


def evaluate_cryptographic_fitness(individual, adaptive_ops):
    """Wrapper to bridge DEAP with our JIT engines, using original NL/DU logic."""
    sbox_arr = np.array(individual, dtype=np.int32)
    
    # ORIGINAL MEASUREMENT LOGIC - unchanged
    min_nl, count_at_min = evaluate_nl_fast(sbox_arr)
    du, _ = compute_differential_uniformity_fast(sbox_arr)
    
    # Maximize NL, Minimize DU, Minimize Weak Count with adaptive weights
    weights = adaptive_ops.get_weights()
    return (min_nl * weights[0], -du * weights[1], -count_at_min * weights[2])


def constrain_repair_bijection(individual):
    """Repair individual to ensure bijection (Point 6)."""
    sbox_array = np.array(individual, dtype=np.int32)
    
    # Find duplicates
    seen = {}
    duplicates = []
    missing = []
    
    for i in range(256):
        if sbox_array[i] in seen:
            duplicates.append(i)
        else:
            seen[sbox_array[i]] = i
    
    # Find missing values
    for val in range(256):
        if val not in seen:
            missing.append(val)
    
    # Repair by swapping duplicates with missing values
    for dup_idx, miss_val in zip(duplicates, missing):
        sbox_array[dup_idx] = miss_val
    
    return list(sbox_array)


def memetic_targeted_mutation_advanced(individual, adaptive_ops, phase="exploitation"):
    """
    Advanced DDT-Aware Mutation with NL-focused exploitation phase.
    Prioritize positions affecting weak bent functions.
    """
    if random.random() < 0.65 and phase == "exploitation":
        # Phase 1: Exploitation - NL-FOCUSED mutation
        sbox_arr = np.array(individual, dtype=np.int32)
        min_nl, count_at_min = evaluate_nl_fast(sbox_arr)
        max_du, ddt = compute_differential_uniformity_fast(sbox_arr)
        
        if count_at_min > 0 and count_at_min < 20:
            # We have very few weak bent functions - mutate strategically
            # Hunt positions that affect multiple output bits
            mutation_targets = []
            for x in range(256):
                # Count how many different bits differ from neighbors
                diff_count = 0
                for bit in range(8):
                    neighbor = x ^ (1 << bit)
                    if individual[x] != individual[neighbor]:
                        diff_count += 1
                mutation_targets.append((x, diff_count))
            
            # Prioritize positions with low bit differences (high rigidity)
            mutation_targets.sort(key=lambda t: t[1])
            
            if mutation_targets:
                # Mutate low-rigidity positions
                for _ in range(random.randint(2, 5)):
                    x, _ = mutation_targets[random.randint(0, min(20, len(mutation_targets)-1))]
                    new_x = random.randint(0, 255)
                    individual[x], individual[new_x] = individual[new_x], individual[x]
        else:
            # Standard exploitation
            idx1, idx2 = random.sample(range(256), 2)
            individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
    else:
        # Phase 2: Exploration - aggressive multi-bit mutations
        num_mutations = random.randint(2, 6)
        for _ in range(num_mutations):
            if random.random() < 0.6:
                # Swap mutation
                idx1, idx2 = random.sample(range(256), 2)
                individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
            else:
                # Bit-flip mutation with 2-3 bits
                idx = random.randint(0, 255)
                num_bits = random.randint(1, 3)
                for _ in range(num_bits):
                    bit_pos = random.randint(0, 7)
                    individual[idx] ^= (1 << bit_pos)
    
    # Ensure bijection
    individual[:] = constrain_repair_bijection(individual)
    return individual,


def diversity_aware_crossover(ind1, ind2, adaptive_ops):
    """Perform crossover only if parents are sufficiently diverse (Point 5)."""
    hamming = adaptive_ops.calculate_hamming_distance(ind1, ind2)
    
    # More aggressive crossover in early phase for exploration
    min_diversity_threshold = 0.05 if adaptive_ops.phase == "early" else 0.08
    
    if hamming > min_diversity_threshold or adaptive_ops.population_diversity_history[-1] < 0.10:
        # Use order crossover
        size = len(ind1)
        a, b = random.sample(range(size), 2)
        if a > b:
            a, b = b, a
        
        child1 = ind1[:]
        child2 = ind2[:]
        
        child1[a:b], child2[a:b] = child2[a:b], child1[a:b]
        
        # Repair bijections
        child1 = constrain_repair_bijection(child1)
        child2 = constrain_repair_bijection(child2)
        
        del ind1.fitness.values
        del ind2.fitness.values
        ind1[:] = child1
        ind2[:] = child2
        return True
    return False


def intense_local_search_nl_focused(individual, adaptive_ops, toolbox, max_attempts=500):
    """
    Intensive local search specifically for NL optimization.
    Uses multi-phase approach targeting weak bent function positions.
    """
    sbox_arr = np.array(individual, dtype=np.int32)
    original_fitness = individual.fitness.values
    original_nl = original_fitness[0] / adaptive_ops.current_weights[0]
    
    improved = False
    stagnation = 0
    max_stagnation = 50
    
    for attempt in range(max_attempts):
        if stagnation > max_stagnation:
            break
        
        # Adaptive neighborhood size based on attempt
        if attempt < 100:
            # Intensive phase: 2-3 swaps
            num_ops = random.randint(2, 3)
        elif attempt < 300:
            # Moderate phase: 1-2 swaps
            num_ops = random.randint(1, 2)
        else:
            # Fine-tuning: single swaps
            num_ops = 1
        
        # Store current state
        backup_ind = individual[:]
        
        # Apply mutations
        for _ in range(num_ops):
            idx1, idx2 = random.sample(range(256), 2)
            individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
        
        # Evaluate
        new_fitness = evaluate_cryptographic_fitness(individual, adaptive_ops)
        new_nl = new_fitness[0] / adaptive_ops.current_weights[0]
        
        # Accept if NL improves OR maintains NL with better DU/weak-count
        nl_improved = new_nl > original_nl
        multi_obj_better = (new_fitness[1] >= original_fitness[1] and 
                            new_fitness[2] >= original_fitness[2])
        
        if nl_improved or (new_nl == original_nl and multi_obj_better):
            individual.fitness.values = new_fitness
            improved = True
            original_fitness = new_fitness
            original_nl = new_nl
            stagnation = 0
        else:
            # Revert
            individual[:] = backup_ind
            stagnation += 1
    
    return improved


def lamarckian_local_search(individual, adaptive_ops, toolbox):
    """
    Original local search retained for backward compatibility.
    """
    sbox_arr = np.array(individual, dtype=np.int32)
    original_fitness = individual.fitness.values
    improved = False
    max_attempts = 150
    attempts = 0
    
    for _ in range(max_attempts):
        idx1, idx2 = random.sample(range(256), 2)
        individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
        
        new_fitness = evaluate_cryptographic_fitness(individual, adaptive_ops)
        
        any_better = any(nf > of for nf, of in zip(new_fitness, original_fitness))
        any_worse = any(nf < of for nf, of in zip(new_fitness, original_fitness))
        
        if any_better and not any_worse:
            individual.fitness.values = new_fitness
            improved = True
            original_fitness = new_fitness
            attempts = 0
        else:
            individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
            attempts += 1
    
    return improved


# Base AES S-Box
AES_SBOX = [
    0x19, 0x10, 0x21, 0xCF, 0x3B, 0x0B, 0xFA, 0x53, 0xB7, 0x2A, 0xB0, 0x17, 0xD5, 0x93, 0x70, 0x05,
    0xC6, 0xC5, 0xBD, 0x3A, 0x71, 0xB8, 0x96, 0x6A, 0x76, 0xE8, 0xE3, 0x27, 0xEB, 0x16, 0xAF, 0xDD,
    0xFC, 0xAE, 0x7E, 0xD3, 0x42, 0x22, 0xB5, 0x33, 0x13, 0x3C, 0x75, 0x40, 0xD4, 0x06, 0x9D, 0x1F,
    0xA4, 0x02, 0x5D, 0xA6, 0xDB, 0xF0, 0x8E, 0x1E, 0xDF, 0xC0, 0x94, 0xAA, 0xCA, 0xF9, 0x72, 0x4E,
    0x60, 0xEF, 0xC8, 0x8A, 0xA0, 0xD0, 0xC3, 0xB2, 0x89, 0x86, 0xB9, 0x58, 0x46, 0x80, 0xB3, 0x30,
    0xA3, 0x66, 0x00, 0x3F, 0x26, 0x84, 0x09, 0xE9, 0x74, 0xEC, 0x9C, 0xD7, 0x52, 0x08, 0x92, 0x48,
    0x4C, 0x6E, 0xA9, 0x5B, 0x32, 0x8D, 0xCC, 0x1A, 0xC7, 0x0A, 0x51, 0x6B, 0xD8, 0x59, 0x90, 0xF8,
    0xF2, 0xC9, 0x49, 0x01, 0x54, 0xC1, 0xFD, 0xE4, 0xCD, 0xD6, 0x57, 0x0C, 0x91, 0xFF, 0x11, 0x1D,
    0x14, 0xEE, 0xEA, 0x15, 0x4D, 0xDA, 0xED, 0x97, 0x79, 0x6D, 0x41, 0x73, 0xCB, 0x1C, 0xF1, 0x85,
    0x6F, 0xBE, 0xDC, 0xA2, 0x77, 0x3D, 0x95, 0xDE, 0xBC, 0x44, 0x69, 0x35, 0xF3, 0x67, 0x31, 0x7D,
    0xFB, 0xE6, 0xAC, 0x1B, 0x29, 0xF5, 0x82, 0x36, 0x8C, 0x37, 0x5C, 0x3E, 0x12, 0x8B, 0xA7, 0x9E,
    0x24, 0x5F, 0x68, 0xAB, 0x50, 0xC2, 0xF6, 0xD2, 0x81, 0x2D, 0xCE, 0xE0, 0xE1, 0xE5, 0x0D, 0x4A,
    0x38, 0xB4, 0xA8, 0x99, 0x7F, 0x47, 0x87, 0x28, 0xB1, 0x88, 0x5A, 0x04, 0x78, 0xE7, 0xA5, 0x8F,
    0xFE, 0x5E, 0xAD, 0x9A, 0x03, 0x7B, 0x9F, 0xBF, 0x45, 0x39, 0x07, 0x43, 0x61, 0xF7, 0x55, 0xBA,
    0xD1, 0x20, 0x4F, 0xBB, 0x0F, 0x2C, 0x2B, 0xA1, 0x34, 0x2E, 0x4B, 0x9B, 0x62, 0x56, 0x6C, 0xD9,
    0x7A, 0x98, 0xF4, 0xC4, 0xB6, 0x0E, 0x18, 0x25, 0x63, 0x65, 0xE2, 0x2F, 0x23, 0x7C, 0x83, 0x64
]

def generate_robust_seed(sbox):
    """Applies a bitwise rotation AND an XOR mask. Guarantees bijection preservation."""
    mask = random.randint(1, 255)
    shift = random.randint(1, 7)
    seeded = []
    for x in sbox:
        rotated = ((x << shift) | (x >> (8 - shift))) & 0xFF
        seeded.append(rotated ^ mask)
    return seeded


def calculate_population_diversity(population):
    """Calculate population diversity metric (0-1)."""
    if len(population) < 2:
        return 0.0
    
    # Sample diversity via pairwise Hamming distances
    sample_size = min(25, len(population) // 2)
    if sample_size < 2:
        return 0.0
    
    total_distance = 0.0
    comparisons = 0
    
    for _ in range(sample_size):
        ind1, ind2 = random.sample(population, 2)
        hamming = sum(a != b for a, b in zip(ind1, ind2)) / 256
        total_distance += hamming
        comparisons += 1
    
    return total_distance / comparisons if comparisons > 0 else 0.0


# ==========================================
# PART 3: MAIN EVOLUTIONARY PIPELINE (Advanced)
# ==========================================

def run_memetic_algorithm_advanced(pop_size=1000, n_gen=500, num_islands=3):
    """
    Memetic Algorithm - Focus on NL optimization.

    - NL-focused fitness weights (8.0 in early phase)
    - Intensive local search with max 500 attempts
    - More aggressive mutation rates (up to 0.95)
    - Reduced stagnation tolerance (smaller improvement threshold)
    - Archive seeding from best NL individuals
    - Extended late-phase refinement
    """
    
    # Initialize fitness and individual classes
    creator.create("FitnessCrypto", base.Fitness, weights=(6.0, 2.0, 0.5))
    creator.create("Individual", list, fitness=creator.FitnessCrypto)
    
    # Initialize toolbox
    toolbox = base.Toolbox()
    
    # Adaptive genetic operators
    adaptive_ops = AdaptiveGeneticOperators()
    
    # Elite archive (Point 9)
    elite_archive = EliteArchive(max_size=150)  # IMPROVEMENT: Larger archive
    
    # Tabu memory (Point 11)
    tabu_memory = TabuMemory(max_size=100)
    
    # Island model (Point 12)
    islands = IslandPopulation(num_islands=num_islands, pop_per_island=pop_size // num_islands)
    
    # Generation phase tracking for seeding (Point 10)
    seeding_schedule = {
        'early': (0, 350, 0.13, 0.87),      # Early: mostly random for exploration
    }
    
    def get_seeding_ratio(gen):
        """Get current seeding ratio based on generation (Point 10)."""
        for _, (start, end, seed_ratio, rand_ratio) in seeding_schedule.items():
            if start <= gen <= end:
                return seed_ratio, rand_ratio
        return 0.6, 0.4
    
    def init_memetic_individual_adaptive(gen):
        """Adaptive seeding based on generation phase (Point 10)."""
        seed_ratio, rand_ratio = get_seeding_ratio(gen)
        
        if random.random() < seed_ratio and elite_archive.archive and random.random() < 0.7:
            #Seed from best NL individuals in archive
            return creator.Individual(elite_archive.get_best_by_nl() or 
                                      generate_robust_seed(AES_SBOX))
        elif random.random() < 0.5:
            return creator.Individual(generate_robust_seed(AES_SBOX))
        else:
            return creator.Individual(random.sample(range(256), 256))
    
    toolbox.register("individual", lambda: init_memetic_individual_adaptive(adaptive_ops.generation))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", lambda ind: evaluate_cryptographic_fitness(ind, adaptive_ops))
    toolbox.register("select", tools.selNSGA2)
    
    # Initial population
    population = toolbox.population(n=pop_size)
    islands.set_islands(population)
    
    print("\n[*] Compiling JIT Engines... (This takes a few seconds on the first generation)")
    
    # Evaluate initial population
    for island_idx, island in enumerate(islands.islands):
        invalid_ind = [ind for ind in island if not ind.fitness.valid]
        for ind in invalid_ind:
            ind.fitness.values = toolbox.evaluate(ind)
        
        # Add to archive
        for ind in island:
            elite_archive.add(ind, ind.fitness.values)
    
    print(f"[*] Engines Compiled. Commencing Multi-Island Memetic Evolution ({n_gen} Generations)...")
    print("-" * 100)
    
    stagnation_counter = 0
    last_best_fitness = None
    nl_plateau_counter = 0
    
    for gen in range(1, n_gen + 1):
        # Periodic migration between islands (Point 12)
        if gen > 1 and gen % islands.migration_interval == 0 and num_islands > 1:
            islands.migrate(elite_ratio=0.08)  # IMPROVEMENT: More aggressive migration
            print(f"[MIGRATION] Generation {gen}: Elite individuals migrated between islands")
        
        # Update adaptive parameters (Points 1, 3)
        all_population = islands.get_all_individuals()
        pop_diversity = calculate_population_diversity(all_population)
        best_ind = max(all_population, key=lambda x: x.fitness.values)
        
        # Denormalize fitness for reporting
        best_fitness_nl = best_ind.fitness.values[0] / adaptive_ops.current_weights[0]
        
        adaptive_ops.update_adaptive_parameters(
            gen, pop_diversity, best_fitness_nl, len(all_population)
        )
        
        # Process each island
        for island_idx, island in enumerate(islands.islands):
            # Selection
            offspring = toolbox.select(island, len(island))
            offspring = list(map(toolbox.clone, offspring))
            
            # Adaptive crossover (Point 5)
            for i in range(0, len(offspring) - 1, 2):
                if random.random() < adaptive_ops.get_crossover_rate():
                    diversity_aware_crossover(offspring[i], offspring[i+1], adaptive_ops)
            
            # Adaptive mutation with hybrid strategy (Points 3, 4)
            for mutant in offspring:
                if random.random() < adaptive_ops.get_mutation_rate():
                    phase = "exploration" if random.random() < 0.3 else "exploitation"
                    memetic_targeted_mutation_advanced(mutant, adaptive_ops, phase=phase)
                    del mutant.fitness.values
            
            # Evaluate offspring
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            for ind in invalid_ind:
                ind.fitness.values = toolbox.evaluate(ind)
            
            # Intensive local search more frequently with NL focus
            ls_frequency = 3 if adaptive_ops.phase == "early" else (2 if adaptive_ops.phase == "mid" else 4)
            if gen % ls_frequency == 0:
                for idx, ind in enumerate(offspring[:20]):  # Apply to top 20
                    if random.random() < 0.4:
                        intense_local_search_nl_focused(ind, adaptive_ops, toolbox, max_attempts=400)
            
            # Replacement with elitism and diversity (Points 2, 9)
            combined = island + offspring
            island[:] = toolbox.select(combined, len(island))
            
            # Add to archive
            for ind in island:
                elite_archive.add(ind, ind.fitness.values)
        
        # Restart logic with higher threshold for NL
        if gen > 80:
            if last_best_fitness is not None and best_fitness_nl - last_best_fitness < 0.3:
                nl_plateau_counter += 1
            else:
                nl_plateau_counter = 0
            
            # More lenient restart (only at 25 gens of plateau)
            if nl_plateau_counter >= 25:
                print(f"[RESTART] Generation {gen}: NL plateau detected, triggering adaptive restart")
                all_population = islands.get_all_individuals()
                
                # Keep top 10%
                all_population.sort(key=lambda x: x.fitness.values, reverse=True)
                num_keep = max(2, int(len(all_population) * 0.10))
                elite_keepers = [toolbox.clone(ind) for ind in all_population[:num_keep]]
                
                # Add elites to tabu list
                for elite in elite_keepers:
                    tabu_memory.add(tuple(elite))
                
                # Regenerate 90%
                new_population = elite_keepers
                while len(new_population) < pop_size:
                    if random.random() < 0.6 and elite_archive.archive:
                        candidate = elite_archive.get_best_by_nl()
                        if candidate and not tabu_memory.is_tabu(tuple(candidate)):
                            new_population.append(creator.Individual(candidate))
                        else:
                            new_population.append(
                                creator.Individual(generate_robust_seed(AES_SBOX))
                            )
                    else:
                        new_population.append(
                            creator.Individual(random.sample(range(256), 256))
                        )
                
                islands.set_islands(new_population)
                
                # Evaluate restarted population
                for island in islands.islands:
                    invalid_ind = [ind for ind in island if not ind.fitness.valid]
                    for ind in invalid_ind:
                        ind.fitness.values = toolbox.evaluate(ind)
                
                nl_plateau_counter = 0
        
        last_best_fitness = best_fitness_nl
        
        # Periodic reporting
        all_pop = islands.get_all_individuals()
        best_ind = max(all_pop, key=lambda x: x.fitness.values)
        
        # Denormalize fitness for reporting
        current_nl = best_ind.fitness.values[0] / adaptive_ops.current_weights[0]
        current_du = -(best_ind.fitness.values[1] / adaptive_ops.current_weights[1])
        
        print(f"Gen {gen:03d} | NL: {current_nl:6.2f} | DU: {current_du:5.1f} | Div: {pop_diversity:.3f} | "
              f"Mut: {adaptive_ops.current_mut_pb:.2f} | CX: {adaptive_ops.current_cx_pb:.2f} | "
              f"Archive: {len(elite_archive.archive)} | Phase: {adaptive_ops.phase}")
        
        # Adaptive milestone output
        if gen % 50 == 0:
            print(f"  -> Archive size: {len(elite_archive.archive)} | "
                  f"Tabu size: {len(tabu_memory.tabu_list)} | "
                  f"NL Plateau: {nl_plateau_counter}")
    
    # ==========================================
    # Output both All-Time Best and Last Round Best
    # ==========================================
    
    # 1. Get the best from the LAST generation
    all_pop = islands.get_all_individuals()
    last_gen_champion = max(all_pop, key=lambda x: x.fitness.values)
    
    # Extract original fitness values for the LAST generation champion
    last_sbox_arr = np.array(last_gen_champion, dtype=np.int32)
    last_nl, _ = evaluate_nl_fast(last_sbox_arr)
    last_du, _ = compute_differential_uniformity_fast(last_sbox_arr)

    # 2. Get the ALL-TIME champion (Archive + Population)
    if elite_archive.archive:
        final_candidates = all_pop + [ind for ind, _ in elite_archive.archive]
    else:
        final_candidates = all_pop
    
    overall_champion = max(final_candidates, key=lambda x: x.fitness.values)
    
    # Extract original fitness values for the OVERALL champion
    best_sbox_arr = np.array(overall_champion, dtype=np.int32)
    best_nl, _ = evaluate_nl_fast(best_sbox_arr)
    best_du, _ = compute_differential_uniformity_fast(best_sbox_arr)
    
    return list(overall_champion), best_nl, best_du, list(last_gen_champion), last_nl, last_du


if __name__ == "__main__":
    # Run advanced memetic algorithm with improved parameters
    best_sbox, best_nl, best_du, last_sbox, last_nl, last_du = run_memetic_algorithm_advanced(
        pop_size=5000, n_gen= 350, num_islands=4
    )
    
    print("\n" + "="*100)
    print("            ADVANCED MEMETIC EVOLUTION COMPLETE             ")
    print("="*100)
    
    # Print Overall Best
    print(f"\n[+] --- ALL-TIME BEST S-BOX (From Archive) ---")
    print(f"[+] Achieved Non-Linearity: {best_nl} (Best Target: 108-112)")
    print(f"[+] Achieved Differential Uniformity: {best_du} (Best Target: 4-6)")
    print(f"Matrix:")
    for row in range(16):
        row_vals = [f"0x{best_sbox[row * 16 + col]:02X}" for col in range(16)]
        print(", ".join(row_vals) + ",")
        
    # Print Last Round Best
    print(f"\n[+] --- LAST GENERATION'S BEST S-BOX (Round 1000) ---")
    print(f"[+] Achieved Non-Linearity: {last_nl} (Best Target: 108-112)")
    print(f"[+] Achieved Differential Uniformity: {last_du} (Best Target: 4-6)")
    print(f"Matrix:")
    for row in range(16):
        row_vals = [f"0x{last_sbox[row * 16 + col]:02X}" for col in range(16)]
        print(", ".join(row_vals) + ",")
        
    print("\n" + "="*100)

/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessCrypto' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "



[*] Compiling JIT Engines... (This takes a few seconds on the first generation)
[*] Engines Compiled. Commencing Multi-Island Memetic Evolution (350 Generations)...
----------------------------------------------------------------------------------------------------
Gen 001 | NL: 106.00 | DU:   6.0 | Div: 0.996 | Mut: 0.80 | CX: 0.12 | Archive: 110 | Phase: early
Gen 002 | NL: 106.00 | DU:   6.0 | Div: 0.997 | Mut: 0.75 | CX: 0.12 | Archive: 150 | Phase: early
Gen 003 | NL: 106.00 | DU:   6.0 | Div: 0.994 | Mut: 0.70 | CX: 0.12 | Archive: 150 | Phase: early
Gen 004 | NL: 106.00 | DU:   6.0 | Div: 0.997 | Mut: 0.65 | CX: 0.12 | Archive: 150 | Phase: early
Gen 005 | NL: 106.00 | DU:   6.0 | Div: 0.997 | Mut: 0.60 | CX: 0.12 | Archive: 150 | Phase: early
Gen 006 | NL: 106.00 | DU:   6.0 | Div: 0.997 | Mut: 0.60 | CX: 0.12 | Archive: 150 | Phase: early
Gen 007 | NL: 106.00 | DU:   6.0 | Div: 0.998 | Mut: 0.60 | CX: 0.12 | Archive: 150 | Phase: early
Gen 008 | NL: 106.00 | DU:   6.0 | Div: 